In [1]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import re
from sklearn.feature_extraction.text import TfidfVectorizer

In [2]:
df = pd.read_csv('../data/sg-tech-jobs-with-salary.csv')
df

,bulletPoints/0,bulletPoints/1,bulletPoints/2,classifications/0/main,classifications/0/sub,company/description,company/id,company/logo,company/name,country,...,salary,scrapedAt,sourceUrl,title,workArrangement,workTypes/0,workTypes/1,workTypes/2,salary_lower,salary_upper
0,NaN,NaN,NaN,Information & Communication Technology,Help Desk & IT Support,PERSOL,60172709.0,https://bx-branding-gateway.cloud.seek.com.au/...,PERSOL,SG,...,"$3,000 – $3,500 per month",2026-02-12T07:38:56.418Z,https://sg.jobstreet.com/jobs-in-information-c...,IT Support Executive,NaN,Full time,NaN,NaN,3000.0,3500.0
1,"5 yrs of exp. in designing, implementing, main...",Dip/Deg in Information Technology/Computer Eng...,CCNA/CCNP/HCIA/HCIP cert. are highly desirable,Information & Communication Technology,Engineering - Network,SmartHire by SEEK,61571544.0,https://bx-branding-gateway.cloud.seek.com.au/...,Smarthire by SEEK,SG,...,"$6,000 – $9,000 per month",2026-02-12T07:38:56.418Z,https://sg.jobstreet.com/jobs-in-information-c...,Network Engineer / Senior Network Engineer,NaN,Full time,NaN,NaN,6000.0,9000.0
2,Diploma/Degree in IT/Computer Science or relat...,Experience in managing networks and understand...,"Proficiency in ERP systems, scripting, and SQL...",Information & Communication Technology,Help Desk & IT Support,SmartHire by SEEK,61571544.0,https://bx-branding-gateway.cloud.seek.com.au/...,Smarthire by SEEK,SG,...,"$2,800 – $3,500 per month",2026-02-12T07:38:56.418Z,https://sg.jobstreet.com/jobs-in-information-c...,Assistant IT Engineer (L1 Support // Semicondu...,NaN,Full time,NaN,NaN,2800.0,3500.0
3,"Comprehensive healthcare coverage (Up to $1,80...","Fostering (DEI), Diversity, Equity, and Inclus...",Employee Assistance Program,Information & Communication Technology,Help Desk & IT Support,Seagate International Headquarters Pte. Ltd.,60467326.0,https://bx-branding-gateway.cloud.seek.com.au/...,Seagate Technology,SG,...,"$4,200 – $5,200 per month",2026-02-12T07:38:56.418Z,https://sg.jobstreet.com/jobs-in-information-c...,IT Analyst (End User Services),NaN,Full time,NaN,NaN,4200.0,5200.0
4,NaN,NaN,NaN,Information & Communication Technology,Networks & Systems Administration,Caton Technology Asia Pte. Ltd.,63402287.0,NaN,Caton Technology Asia,SG,...,"$4,000 – $5,000 per month",2026-02-12T07:38:56.418Z,https://sg.jobstreet.com/jobs-in-information-c...,Technical Solution Engineer,NaN,Full time,NaN,NaN,4000.0,5000.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3393,NaN,NaN,NaN,Information & Communication Technology,Engineering - Software,Private Advertiser,NaN,NaN,Private Advertiser,SG,...,"$8,000 – $12,000 per month",2026-02-12T08:23:49.116Z,https://sg.jobstreet.com/jobs-in-information-c...,DH Solution Consulting Engineer (FT Handler),NaN,Full time,NaN,NaN,8000.0,12000.0
3394,NaN,NaN,NaN,Information & Communication Technology,Developers/Programmers,Certis Group,60178895.0,https://bx-branding-gateway.cloud.seek.com.au/...,Certis,SG,...,"$4,000 – $6,000 per month",2026-02-12T08:23:49.116Z,https://sg.jobstreet.com/jobs-in-information-c...,RPA Developer,NaN,Full time,NaN,NaN,4000.0,6000.0
3395,NaN,NaN,NaN,Information & Communication Technology,Sales - Pre & Post,APBA TG Human Resource Pte Ltd,60526916.0,https://bx-branding-gateway.cloud.seek.com.au/...,TG Group Pte Ltd,SG,...,"$4,000 – $6,000 per month",2026-02-12T08:23:49.116Z,https://sg.jobstreet.com/jobs-in-information-c...,Sales Engineer (GPU Solutions & Hardware),NaN,Full time,NaN,NaN,4000.0,6000.0
3396,NaN,NaN,NaN,Information & Communication Technology,Database Development & Administration,RecruitFirst Pte. Ltd,60173576.0,https://bx-branding-gateway.cloud.seek.com.au/...,RecruitFirst,SG,...,"$6,000 per month",2026-02-12T08:23:49.116Z,https://sg.jobstreet.com/jobs-in-information-c...,"Data Engineer *SQL - UP $6000, Ubi",NaN,Contract/Temp,NaN,NaN,6000.0,NaN


In [3]:
df['classifications/0/sub'].value_counts()

classifications/0/sub
Help Desk & IT Support                   555
Developers/Programmers                   449
Networks & Systems Administration        445
Engineering - Software                   397
Business/Systems Analysts                347
Programme & Project Management           283
Security                                 170
Management                               109
Testing & Quality Assurance              109
Sales - Pre & Post                        81
Engineering - Network                     75
Database Development & Administration     71
Architects                                71
Product Management & Development          64
Other                                     46
Engineering - Hardware                    45
Consultants                               40
Telecommunications                        13
Team Leaders                              12
Web Development & Production               9
Technical Writing                          6
Computer Operators               

In [4]:
df['workTypes/0'].value_counts()

workTypes/0
Full time        2332
Contract/Temp    1054
Part time          12
Name: count, dtype: int64

In [5]:
df['location'].value_counts()

location
Central Region                     1271
West Region                         259
East Region                         257
Singapore River, Central Region     109
Paya Lebar, East Region              85
                                   ... 
Geylang East, Central Region          3
East Coast, Central Region            3
Kent Ridge, Central Region            3
Telok Blangah, Central Region         3
Shipyard, West Region                 3
Name: count, Length: 99, dtype: int64

In [6]:
df['location'] = df['location'].apply(lambda x: re.findall(r'([A-Z][a-z]+) Region', x))

In [7]:
df['location'] = df['location'].apply(lambda x: x[0] if x else 'Unknown')
df['location'].value_counts()

location
Central      1963
East          834
West          448
North         123
Unknown        24
Woodlands       6
Name: count, dtype: int64

Map estate to region.

In [8]:
df['location'] = np.where(df['location'] == 'Woodlands', 'North', df['location'])

In [9]:
df['location'].value_counts()

location
Central    1963
East        834
West        448
North       129
Unknown      24
Name: count, dtype: int64

In [10]:
df['text_corpus'] = df['title'].fillna(' ') + ' ' + df['company/description'].fillna('') + ' ' + df['bulletPoints/0'].fillna('') + ' ' + df['bulletPoints/1'].fillna('') + ' ' + df['bulletPoints/2'].fillna('')
df['text_corpus'] = df['text_corpus'].str.lower()
df['text_corpus']

0                          it support executive persol   
1       network engineer / senior network engineer sma...
2       assistant it engineer (l1 support // semicondu...
3       it analyst (end user services) seagate interna...
4       technical solution engineer caton technology a...
                              ...                        
3393    dh solution consulting engineer (ft handler) p...
3394                        rpa developer certis group   
3395    sales engineer (gpu solutions & hardware) apba...
3396    data engineer *sql - up $6000, ubi recruitfirs...
3397    (japanese speaking) infrastructure engineer, i...
Name: text_corpus, Length: 3398, dtype: str

In [11]:
print(df['text_corpus'].iloc[1])

network engineer / senior network engineer smarthire by seek 5 yrs of exp. in designing, implementing, maintaining complex network infras. dip/deg in information technology/computer engineering or relevant certification ccna/ccnp/hcia/hcip cert. are highly desirable


# Binary Features

This approach finds the presence of several tech keywords from the text corpus and flag them if found.

In [12]:
keywords = {
    'python': r'python',
    'llm': r'llm|large language model|gpt|generative ai|genai|gen ai',
    'data': r'data|data science|data analytics|data analysis|data engineering|data scientist|data analyst|data engineer',
    'production': r'production|deployment|ship',
    'ai': r' ai |artificial intelligence|machine learning|ml',
    'sql': r'sql|database|postgres|mysql|oracle',
    'cloud': r'aws|azure|cloud|google cloud|gcp',
    'senior': r'senior|lead|manager|head|principal|sr\.',
    'java': r'java[^script]',
    'javascript': r'javascript|react|node|typescript',
    'security': r'security|cyber|infosec',
    'git': r'git|github|gitlab',
    'software_engineer': r'software engineer|software developer|backend|frontend|back end|front end|full[- ]?stack',
    'communication': r'communication|communicating',
    'proficient': r'proficient',
    'competitive': r'competitive|attractive|bonus|appraisal',
    'healthcare': r'healthcare|health|insurance',
    'consulting': r'consulting|consultant',
    'bank_finance': r'bank|banking|finance|financial',
    'contract': r'contract',
    'renewable': r'renewable',
    'hybrid': r'hybrid'
}

for keyword, pattern in keywords.items():
    df[keyword] = df['text_corpus'].str.contains(pattern, regex=True).astype(int)

In [13]:
keyword_cols = list(keywords.keys())
categorical_cols = ['location', 'classifications/0/sub', 'workTypes/0']

In [14]:
df_clean = df[keyword_cols + categorical_cols + ['salary_lower', 'salary_upper']]
df_clean

,python,llm,data,production,ai,sql,cloud,senior,java,javascript,...,consulting,bank_finance,contract,renewable,hybrid,location,classifications/0/sub,workTypes/0,salary_lower,salary_upper
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,East,Help Desk & IT Support,Full time,3000.0,3500.0
1,0,0,0,0,0,0,0,1,0,0,...,0,0,0,0,0,Central,Engineering - Network,Full time,6000.0,9000.0
2,0,0,0,0,0,1,0,0,0,0,...,0,0,0,0,0,East,Help Desk & IT Support,Full time,2800.0,3500.0
3,0,0,0,0,0,0,0,1,0,0,...,0,0,0,0,0,North,Help Desk & IT Support,Full time,4200.0,5200.0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,Central,Networks & Systems Administration,Full time,4000.0,5000.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3393,0,0,0,0,0,0,0,0,0,0,...,1,0,0,0,0,Central,Engineering - Software,Full time,8000.0,12000.0
3394,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,Central,Developers/Programmers,Full time,4000.0,6000.0
3395,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,East,Sales - Pre & Post,Full time,4000.0,6000.0
3396,0,0,1,0,0,1,0,0,0,0,...,0,0,0,0,0,Central,Database Development & Administration,Contract/Temp,6000.0,NaN


In [15]:
df_clean.to_csv('../data/sg-tech-jobs-preprocessed-keywords_v2.csv', index=False)

# Text Corpus + Work Types + Location Data

Using rule-based approach to map keywords (above cell) may introduce several drawbacks:
- unable to understand context: `"leading global bank"` mapped to 1 because of the string `"lead"`, which activates the `"senior"` feature
- unable to identify negative meaning: `"this is a not a contract position"` flags `"contract"` feature as true

Therefore, save the data with text corpus to enable using more advanced text preprocessing technique like TF-IDF.

In [16]:
df.head()

,bulletPoints/0,bulletPoints/1,bulletPoints/2,classifications/0/main,classifications/0/sub,company/description,company/id,company/logo,company/name,country,...,software_engineer,communication,proficient,competitive,healthcare,consulting,bank_finance,contract,renewable,hybrid
0,NaN,NaN,NaN,Information & Communication Technology,Help Desk & IT Support,PERSOL,60172709.0,https://bx-branding-gateway.cloud.seek.com.au/...,PERSOL,SG,...,0,0,0,0,0,0,0,0,0,0
1,"5 yrs of exp. in designing, implementing, main...",Dip/Deg in Information Technology/Computer Eng...,CCNA/CCNP/HCIA/HCIP cert. are highly desirable,Information & Communication Technology,Engineering - Network,SmartHire by SEEK,61571544.0,https://bx-branding-gateway.cloud.seek.com.au/...,Smarthire by SEEK,SG,...,0,0,0,0,0,0,0,0,0,0
2,Diploma/Degree in IT/Computer Science or relat...,Experience in managing networks and understand...,"Proficiency in ERP systems, scripting, and SQL...",Information & Communication Technology,Help Desk & IT Support,SmartHire by SEEK,61571544.0,https://bx-branding-gateway.cloud.seek.com.au/...,Smarthire by SEEK,SG,...,0,0,0,0,0,0,0,0,0,0
3,"Comprehensive healthcare coverage (Up to $1,80...","Fostering (DEI), Diversity, Equity, and Inclus...",Employee Assistance Program,Information & Communication Technology,Help Desk & IT Support,Seagate International Headquarters Pte. Ltd.,60467326.0,https://bx-branding-gateway.cloud.seek.com.au/...,Seagate Technology,SG,...,0,0,0,0,1,0,0,0,0,0
4,NaN,NaN,NaN,Information & Communication Technology,Networks & Systems Administration,Caton Technology Asia Pte. Ltd.,63402287.0,NaN,Caton Technology Asia,SG,...,0,0,0,0,0,0,0,0,0,0


In [17]:
categorical_cols = ['location', 'classifications/0/sub', 'workTypes/0']
df_corpus_worktypes_location = df[categorical_cols + ['text_corpus', 'salary_lower', 'salary_upper']]
df_corpus_worktypes_location

,location,classifications/0/sub,workTypes/0,text_corpus,salary_lower,salary_upper
0,East,Help Desk & IT Support,Full time,it support executive persol,3000.0,3500.0
1,Central,Engineering - Network,Full time,network engineer / senior network engineer sma...,6000.0,9000.0
2,East,Help Desk & IT Support,Full time,assistant it engineer (l1 support // semicondu...,2800.0,3500.0
3,North,Help Desk & IT Support,Full time,it analyst (end user services) seagate interna...,4200.0,5200.0
4,Central,Networks & Systems Administration,Full time,technical solution engineer caton technology a...,4000.0,5000.0
...,...,...,...,...,...,...
3393,Central,Engineering - Software,Full time,dh solution consulting engineer (ft handler) p...,8000.0,12000.0
3394,Central,Developers/Programmers,Full time,rpa developer certis group,4000.0,6000.0
3395,East,Sales - Pre & Post,Full time,sales engineer (gpu solutions & hardware) apba...,4000.0,6000.0
3396,Central,Database Development & Administration,Contract/Temp,"data engineer *sql - up $6000, ubi recruitfirs...",6000.0,NaN


In [18]:
df_corpus_worktypes_location.to_csv('../data/sg-tech-jobs-preprocessed-corpus_worktypes_location.csv', index=False)

# Text Corpus-Only Data

In [19]:
df

,bulletPoints/0,bulletPoints/1,bulletPoints/2,classifications/0/main,classifications/0/sub,company/description,company/id,company/logo,company/name,country,...,software_engineer,communication,proficient,competitive,healthcare,consulting,bank_finance,contract,renewable,hybrid
0,NaN,NaN,NaN,Information & Communication Technology,Help Desk & IT Support,PERSOL,60172709.0,https://bx-branding-gateway.cloud.seek.com.au/...,PERSOL,SG,...,0,0,0,0,0,0,0,0,0,0
1,"5 yrs of exp. in designing, implementing, main...",Dip/Deg in Information Technology/Computer Eng...,CCNA/CCNP/HCIA/HCIP cert. are highly desirable,Information & Communication Technology,Engineering - Network,SmartHire by SEEK,61571544.0,https://bx-branding-gateway.cloud.seek.com.au/...,Smarthire by SEEK,SG,...,0,0,0,0,0,0,0,0,0,0
2,Diploma/Degree in IT/Computer Science or relat...,Experience in managing networks and understand...,"Proficiency in ERP systems, scripting, and SQL...",Information & Communication Technology,Help Desk & IT Support,SmartHire by SEEK,61571544.0,https://bx-branding-gateway.cloud.seek.com.au/...,Smarthire by SEEK,SG,...,0,0,0,0,0,0,0,0,0,0
3,"Comprehensive healthcare coverage (Up to $1,80...","Fostering (DEI), Diversity, Equity, and Inclus...",Employee Assistance Program,Information & Communication Technology,Help Desk & IT Support,Seagate International Headquarters Pte. Ltd.,60467326.0,https://bx-branding-gateway.cloud.seek.com.au/...,Seagate Technology,SG,...,0,0,0,0,1,0,0,0,0,0
4,NaN,NaN,NaN,Information & Communication Technology,Networks & Systems Administration,Caton Technology Asia Pte. Ltd.,63402287.0,NaN,Caton Technology Asia,SG,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3393,NaN,NaN,NaN,Information & Communication Technology,Engineering - Software,Private Advertiser,NaN,NaN,Private Advertiser,SG,...,0,0,0,0,0,1,0,0,0,0
3394,NaN,NaN,NaN,Information & Communication Technology,Developers/Programmers,Certis Group,60178895.0,https://bx-branding-gateway.cloud.seek.com.au/...,Certis,SG,...,0,0,0,0,0,0,0,0,0,0
3395,NaN,NaN,NaN,Information & Communication Technology,Sales - Pre & Post,APBA TG Human Resource Pte Ltd,60526916.0,https://bx-branding-gateway.cloud.seek.com.au/...,TG Group Pte Ltd,SG,...,0,0,0,0,0,0,0,0,0,0
3396,NaN,NaN,NaN,Information & Communication Technology,Database Development & Administration,RecruitFirst Pte. Ltd,60173576.0,https://bx-branding-gateway.cloud.seek.com.au/...,RecruitFirst,SG,...,0,0,0,0,0,0,0,0,0,0


In [22]:
df_corpus_only = df[['text_corpus', 'salary_lower', 'salary_upper']]
df_corpus_only

,text_corpus,salary_lower,salary_upper
0,it support executive persol,3000.0,3500.0
1,network engineer / senior network engineer sma...,6000.0,9000.0
2,assistant it engineer (l1 support // semicondu...,2800.0,3500.0
3,it analyst (end user services) seagate interna...,4200.0,5200.0
4,technical solution engineer caton technology a...,4000.0,5000.0
...,...,...,...
3393,dh solution consulting engineer (ft handler) p...,8000.0,12000.0
3394,rpa developer certis group,4000.0,6000.0
3395,sales engineer (gpu solutions & hardware) apba...,4000.0,6000.0
3396,"data engineer *sql - up $6000, ubi recruitfirs...",6000.0,NaN


In [23]:
df_corpus_only.to_csv('../data/sg-tech-jobs-preprocessed-corpus-only.csv', index=False)